In [17]:
import pandas as pd

# =============================
# Dataset Loading
# =============================

games_df = pd.read_csv("../Data/api data/Old data/Final Database/games.csv")
detailed_games_df = pd.read_csv("../Data/api data/Old data/Final Database/detailed_games.csv")
company_games_df = pd.read_csv("../Data/api data/Old data/Final Database/company_games.csv")
genre_games_df = pd.read_csv("../Data/api data/Old data/Final Database/genre_games.csv")
platform_df = pd.read_csv("../Data/api data/Old data/Final Database/platform.csv")


final_df = pd.read_csv("../Data/Dataset.csv")

C:\Users\jorda\AppData\Local\Temp\ipykernel_58996\1055050788.py:14: DtypeWarning:

Columns (8,25,33) have mixed types. Specify dtype option on import or set low_memory=False.



In [18]:


# Flatten all genre lists and extract unique genres
unique_genres = set(
    g.strip()
    for genre_str in final_df["genres"].dropna()
    if isinstance(genre_str, str)
    for g in genre_str.split(",")
)


# Convert to a sorted list if needed
unique_genres = sorted(unique_genres)

# Print or return them
print(unique_genres)



['Action', 'Adventure', 'Arcade', 'Board Games', 'Card', 'Casual', 'Educational', 'Family', 'Fighting', 'Indie', 'Massively Multiplayer', 'Platformer', 'Puzzle', 'RPG', 'Racing', 'Shooter', 'Simulation', 'Sports', 'Strategy']


In [27]:
import dash
from dash import dcc, html, Input, Output, State, ctx, ALL
import dash_bootstrap_components as dbc
import threading
import webbrowser
import socket
import plotly.express as px
import plotly.graph_objects as go
import squarify

# Create Dash app with Bootstrap theme
app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])
app.title = "Dashboard"

years = list(range(1970, 2025))

# =========================
# Helper component builders
# =========================

def build_top_control(active_tab):
    return dbc.ButtonGroup([
        dbc.Button("Games", id="games-button", n_clicks=0, color="primary" if active_tab == "games" else "secondary"),
        dbc.Button("Companies", id="companies-button", n_clicks=0, color="primary" if active_tab == "companies" else "secondary")
    ], id="top_control", style={"width": "100%"})

def build_search_bar():
    return dbc.Card(
        dbc.CardBody([
            dbc.Input(
                id="search_bar",
                placeholder="Search...",
                type="text",
                style={"marginBottom": "0"}
            )
        ]),
        style={
            "marginTop": "1rem",
            "marginBottom": "8rem",
            "boxShadow": "0px 2px 6px rgba(0,0,0,0.1)",
            "border": "1px solid #ced4da",
            "borderRadius": "0.5rem",
            "backgroundColor": "white"
        }
    )


def build_data_view():
    return html.Div(
        children=[
            html.H1("Main Visualization Area"),
            html.Div(
                html.Div(id="main_graph", style={"width": "100%"}),
                #dcc.Graph(id="main_graph", style={"height": "600px", "width": "100%"}),
                style={
                    "overflowX": "auto",
                    "width": "100%",
                    "paddingBottom": "1rem"
                }
            )
        ],
        id="data_view",
        style={"padding": "2rem"}
    )



def build_games_middle(selected_sub, selected_sort_options, selected_genres):
    options = ["Most Popular", "Genres"]
    buttons = [
        dbc.Button(
            label,
            id={"type": "sub-button", "index": label.lower().replace(" ", "-") + "-sub"},
            color="primary" if selected_sub == label.lower().replace(" ", "-") + "-sub" else "secondary",
            n_clicks=0,
            style={"width": "100%", "marginBottom": "0.5rem"}
        )
        for label in options
    ]

    if selected_sub == "most-popular-sub":
        sort_options = ["Rating", "YouTube", "Twitch", "Added", "Metacritic"]
    elif selected_sub == "genres-sub":
        sort_options = ['Action', 'Adventure', 'Arcade', 'Board Games', 'Card', 
         'Casual', 'Educational', 'Family', 'Fighting', 'Indie',
           'Massively Multiplayer', 'Platformer', 'Puzzle',
             'RPG', 'Racing', 'Shooter', 'Simulation',
               'Sports', 'Strategy']

    sort_buttons = [
        dbc.Button(
            m,
            id={"type": "sort-button", "index": m},
            color="primary" if m in selected_sort_options else "secondary",
            style={"width": "100%", "marginBottom": "0.25rem"}
        )
        for m in sort_options
    ]

    return html.Div([
        dbc.Row([
            dbc.Col(buttons, width=6),
            dbc.Col([
                html.H6("Sort By:"),
                *sort_buttons
            ], width=6)
        ]),

        # ── Release-year slider (unchanged) ────────────────────────────────
        html.Div([
            html.Label("Select Release Year Range", style={"fontWeight": "bold", "marginTop": "1rem"}),
            dcc.RangeSlider(
                id="year-range-slider",
                min=1970,
                max=2024,
                step=1,
                marks={y: str(y) for y in range(1970, 2025, 10)},
                value=[2000, 2020],
                tooltip={"placement": "bottom", "always_visible": False},
                allowCross=False,
                updatemode="mouseup"
            )
        ], style={
            "marginTop": "1.5rem", "padding": "1rem",
            "backgroundColor": "#f1f3f5", "borderRadius": "0.5rem",
            "boxShadow": "inset 0 1px 3px rgba(0,0,0,0.1)", "width": "100%"
        }),

        # ── NEW: number-of-games slider ────────────────────────────────────
        html.Div([
            html.Label("Number of games (3 – 200)", style={"fontWeight": "bold", "marginTop": "1rem"}),
            dcc.Slider(
                id="num-games-slider",
                min=3, max=200, step=1, value=50,                 # default 50
                marks={i: str(i) for i in range(10, 201, 30)},
                updatemode="drag", tooltip={"placement": "bottom", "always_visible": False}
            )
        ], style={
            "marginTop": "1.5rem", "padding": "1rem",
            "backgroundColor": "#f1f3f5", "borderRadius": "0.5rem",
            "boxShadow": "inset 0 1px 3px rgba(0,0,0,0.1)", "width": "100%"
        })
    ])

def build_companies_middle(selected_sub):
    buttons = []
    options = ["Companies", "Publishers"]
    for label in options:
        idx = label.lower().replace(" ", "-") + "-sub"
        color = "primary" if selected_sub == idx else "secondary"
        buttons.append(
            dbc.Button(
                label,
                id={"type": "sub-button", "index": idx},
                color=color,
                outline=False,
                n_clicks=0,
                style={"width": "100%", "marginBottom": "0.5rem"}
            )
        )
    return html.Div(buttons)

def build_sidebar(active_tab, selected_sub, selected_sort_options, selected_genres):
    return html.Div([
        build_top_control(active_tab),
        html.Hr(),
        html.Div(
            id="middle_options",
            children=build_games_middle(selected_sub, selected_sort_options, selected_genres) if active_tab == "games" else build_companies_middle(selected_sub),
            style={"flexGrow": 1, "overflowY": "auto","overflowX": "hidden","paddingTop": "1rem","paddingBottom": "1rem"}
        ),
        html.Hr(),
        build_search_bar()
    ], style={
        "display": "flex",
        "flexDirection": "column",
        "height": "100%",
        "padding": "1rem"
    })


# =========================
# App Layout
# =========================

# Important: Default sidebar must be built immediately at startup!
initial_active_tab = "games"
initial_selected_sub = "most-popular-sub"


app.layout = dbc.Container(
    fluid=True,
    children=[
        dcc.Store(id="active_main_tab", data=initial_active_tab),
        dcc.Store(id="selected_sub_button", data=initial_selected_sub),
        dcc.Store(id="selected_sort_options", data=[]),
        dcc.Store(id="selected_genres", data=[]),
        dcc.Store(id="selected_year_range", data=[1970, 2024]),
        dcc.Store(id="selected_game_id", data=None),  # New store for selected game

        dbc.Row([
            # Sidebar
            dbc.Col(
                id="sidebar",
                children=build_sidebar(initial_active_tab, initial_selected_sub, [], []),
                width=3,
                style={
                    "backgroundColor": "#f8f9fa",
                    "height": "100vh",
                    "padding": 0,
                    "borderRight": "1px solid #dee2e6",
                    "display": "flex",
                    "flexDirection": "column"
                }
            ),

            # Data View
            dbc.Col(
                build_data_view(),
                width=6  # Reduced width to accommodate panel
            ),

            # Right-Side Panel
            dbc.Col(
                html.Div(
                    id="game-details-panel",
                    style={
                        "height": "100vh",
                        "padding": "1rem",
                        "backgroundColor": "#f8f9fa",
                        "borderLeft": "1px solid #dee2e6",
                        "overflowY": "auto"
                    }
                ),
                width=3
            )
        ])
    ]
)

# =========================
# Callbacks
# =========================

@app.callback(
    [Output("active_main_tab", "data"),
     Output("selected_sub_button", "data")],
    [Input("games-button", "n_clicks"),
     Input("companies-button", "n_clicks"),
     Input({"type": "sub-button", "index": ALL}, "n_clicks")],
    [State("active_main_tab", "data"),
     State("selected_sub_button", "data")]
)
def handle_clicks(games_clicks, companies_clicks, sub_clicks, current_tab, selected_sub):
    triggered = ctx.triggered_id

    if triggered == "games-button":
        return "games", "most-popular-sub"
    elif triggered == "companies-button":
        return "companies", "companies-sub"
    elif isinstance(triggered, dict) and triggered.get("type") == "sub-button":
        return current_tab, triggered["index"]
    else:
        return current_tab, selected_sub

@app.callback(
    Output("sidebar", "children"),
    [Input("active_main_tab", "data"),
     Input("selected_sub_button", "data"),
     Input("selected_sort_options", "data"),
     Input("selected_genres", "data")]
)
def update_sidebar(active_tab, selected_sub, selected_sort_options, selected_genres):
    return build_sidebar(active_tab, selected_sub, selected_sort_options, selected_genres)

@app.callback(
    Output("selected_genres", "data"),
    Input({"type": "genre-button", "index": ALL}, "n_clicks"),
    State("selected_genres", "data"),
    prevent_initial_call=True
)
def toggle_genre_selection(n_clicks_list, selected_genres):
    triggered = ctx.triggered_id
    if not triggered:
        return dash.no_update

    genre = triggered["index"]
    if genre in selected_genres:
        selected_genres.remove(genre)
    else:
        selected_genres.append(genre)

    return selected_genres

@app.callback(
    Output("selected_sort_options", "data"),
    Input({"type": "sort-button", "index": ALL}, "n_clicks"),
    State("selected_sort_options", "data"),
    prevent_initial_call=True
)
def select_sort_option(n_clicks_list, selected_sort_options):
    triggered = ctx.triggered_id
    if not triggered:
        return dash.no_update

    sort_option = triggered["index"]
    return [sort_option]

@app.callback(
    Output("selected_year_range", "data"),
    Input("year-range-slider", "value"),
    prevent_initial_call=True
)
def update_selected_year_range(year_range):
    return year_range


@app.callback(
    Output("main_graph", "children"),
    [Input("active_main_tab", "data"),
     Input("selected_sub_button", "data"),
     Input("selected_sort_options", "data"),
     Input("selected_year_range", "data"),
     Input("num-games-slider", "value")]
)
def update_main_graph(active_tab, selected_sub_button, selected_sort_options, selected_year_range, num_games):
    if active_tab == "companies" and selected_sub_button == "companies-sub":
        company_counts = company_games_df["company"].value_counts().reset_index()
        company_counts.columns = ["Company", "Number of Games"]

        company_counts = company_counts.sort_values("Number of Games", ascending=False)

        fig = px.bar(
            company_counts,
            x="Company",
            y="Number of Games",
            title="Top Companies by Number of Games",
            labels={"Company": "Company", "Number of Games": "Number of Games"},
        )

        fig.update_layout(
            xaxis_tickangle=-45,
            height=600,
            margin=dict(l=50, r=30, t=50, b=150),
            bargap=0.2,
        )

        return fig

    elif active_tab == "games" and selected_sub_button == "most-popular-sub":
        # 1. Filter & rank
        df = final_df.copy()
        df = df.dropna(subset=["name"]).drop_duplicates()
        df = df[
            (df["release_year"] >= selected_year_range[0]) &
            (df["release_year"] <= selected_year_range[1])
        ]

        column_mapping = {
            "rating": "rating",
            "youtube": "youtube_count",
            "twitch": "twitch_count",
            "added": "added",
            "metacritic": "metacritic",
        }
        sort_by = column_mapping.get(
            (selected_sort_options[0] if selected_sort_options else "rating").lower(),
            "rating",
        )

        num_games = num_games or 50           # fallback default
        df = (
            df.dropna(subset=[sort_by])
            .sort_values(sort_by, ascending=False)
            .head(num_games)
        )

        # 2. Treemap areas – **linear** with the metric
        #    Guard against zeros so we don't hand 0-area to squarify.
        metric = df[sort_by].astype(float).clip(lower=1e-6)
        areas  = metric                       # <- direct proportionality
        normed = squarify.normalize_sizes(areas, 100, 100)
        rects  = squarify.squarify(normed, 0, 0, 100, 100)
        df = pd.concat([df.reset_index(drop=True), pd.DataFrame(rects)], axis=1)

        # 3. Build tiles (unchanged visual style)
        
        tiles = [
            html.Div(
                children=[
                    html.Div(
                        [
                            html.Div(
                                row["name"],
                                style={
                                    "fontSize": "12px", "fontWeight": "bold",
                                    "overflow": "hidden", "textOverflow": "ellipsis",
                                    "whiteSpace": "nowrap",
                                },
                            ),
                            html.Div(
                                f"{sort_by.capitalize()}: {row[sort_by]:.2f}",
                                style={"fontSize": "10px"},
                            ),
                        ],
                        style={
                            "position": "absolute", "inset": 0,
                            "backgroundColor": "rgba(0,0,0,0.55)",
                            "display": "flex", "flexDirection": "column",
                            "alignItems": "center", "justifyContent": "center",
                            "padding": "4px",
                        },
                    )
                ],
                id={"type": "game-tile", "index": int(row["id"])},  # Add unique ID
                n_clicks=0,  # Enable click events
                style={
                    "position": "absolute",
                    "left": f"{row['x']:.2f}%", "top": f"{row['y']:.2f}%",
                    "width": f"{row['dx']:.2f}%", "height": f"{row['dy']:.2f}%",
                    "backgroundImage": f"url('{row['background_image']}')",
                    "backgroundSize": "cover",
                    "backgroundPosition": "center",
                    "border": "1px solid #fff",
                    "boxSizing": "border-box",
                    "borderRadius": "4px",
                    "overflow": "hidden",
                    "color": "#fff",
                    "cursor": "pointer"  # Indicate clickability
                },
            )
            for _, row in df.iterrows()
        ]

        # 4. Return wrapped container
        return html.Div(
            tiles,
            style={
                "position": "relative",
                "width": "100%",
                "height": "60vh",
                "backgroundColor": "#333",
            },
        )
    
    elif active_tab == "games" and selected_sub_button == "genres-sub":
        # 1. Filter & rank
        df = final_df.copy()
        df = df.dropna(subset=["name"]).drop_duplicates()
        df = df[
            (df["release_year"] >= selected_year_range[0]) &
            (df["release_year"] <= selected_year_range[1])
        ]

        column_mapping = {
            "Action": "Action",
            "Adventure": "Adventure",
            "Arcade": "Arcade",
            "Board Games": "Board Games",
            "Card": "Card",
            "Casual": "Casual",
            "Educational": "Educational",
            "Family": "Family",
            "Fighting": "Fighting",
            "Indie": "Indie",
            "Massively Multiplayer": "Massively Multiplayer",
            "Platformer": "Platformer",
            "Puzzle": "Puzzle",
            "RPG": "RPG",
            "Racing": "Racing",
            "Shooter": "Shooter",
            "Simulation": "Simulation",
            "Sports": "Sports",
            "Strategy": "Strategy",
        }

        if selected_sub_button == "genres-sub":
            valid_genres = list(column_mapping.keys())
            if selected_sort_options and selected_sort_options[0] in valid_genres:
                selected_genre = selected_sort_options[0]
            else:
                return html.Div("No genre selected or recognized.", style={"color": "white", "padding": "1rem"})

            # Define genre extraction from list
            df = df[df["genres"].notna()]
            df = df.explode("genres")
            df = df[df["genres"].isin(column_mapping.keys())]
            df = df.rename(columns={"genres": "genre"})

            df = df[df["genre"] == selected_genre]

            sort_by = "rating"  # static metric for size

            # Rank top N per genre
            df = df[df[sort_by].notna() & (df[sort_by] > 0)].sort_values(sort_by, ascending=False).head(50)

            if df.empty:
                return html.Div("No data available for this genre.", style={"color": "white", "padding": "1rem"})

            normed = squarify.normalize_sizes(df[sort_by], 100, 100)
            rects = squarify.squarify(normed, 0, 0, 100, 100)
            df = pd.concat([df.reset_index(drop=True), pd.DataFrame(rects)], axis=1)

            # Build individual tiles
            # Inside update_main_graph, under elif active_tab == "games" and selected_sub_button == "genres-sub":
            # Replace the tiles creation with:

            tiles = [
                html.Div(
                    children=[
                        html.Div(
                            [
                                html.Div(
                                    row["name"],
                                    style={
                                        "fontSize": "11px",
                                        "fontWeight": "bold",
                                        "overflow": "hidden",
                                        "textOverflow": "ellipsis",
                                        "whiteSpace": "nowrap",
                                    },
                                ),
                                html.Div(
                                    f"{sort_by.capitalize()}: {row[sort_by]:.2f}",
                                    style={"fontSize": "10px"},
                                ),
                            ],
                            style={
                                "position": "absolute",
                                "inset": 0,
                                "backgroundColor": "rgba(0,0,0,0.55)",
                                "display": "flex",
                                "flexDirection": "column",
                                "alignItems": "center",
                                "justifyContent": "center",
                                "padding": "4px",
                            },
                        )
                    ],
                    id={"type": "game-tile", "index": int(row["id"])},  # Add unique ID
                    n_clicks=0,  # Enable click events
                    style={
                        "position": "absolute",
                        "left": f"{row['x']:.2f}%",
                        "top": f"{row['y']:.2f}%",
                        "width": f"{row['dx']:.2f}%",
                        "height": f"{row['dy']:.2f}%",
                        "backgroundImage": f"url('{row['background_image']}')",
                        "backgroundSize": "cover",
                        "backgroundPosition": "center",
                        "border": "1px solid #fff",
                        "boxSizing": "border-box",
                        "borderRadius": "4px",
                        "overflow": "hidden",
                        "color": "#fff",
                        "cursor": "pointer"  # Indicate clickability
                    },
                )
                for _, row in df.iterrows()
            ]

            # Return wrapped container (unchanged)
            return html.Div(
                tiles,
                style={
                    "position": "relative",
                    "width": "100%",
                    "height": "75vh",
                    "backgroundColor": "#333",
                },
            )
    else:
        return px.bar(title="Select a tab to view data")



# Add these callbacks after the existing callbacks in Dashboard.ipynb

@app.callback(
    Output("selected_game_id", "data"),
    Input({"type": "game-tile", "index": ALL}, "n_clicks"),
    State({"type": "game-tile", "index": ALL}, "id"),
    prevent_initial_call=True
)
def handle_tile_click(n_clicks_list, tile_ids):
    triggered = ctx.triggered_id
    if not triggered:
        return dash.no_update
    game_id = triggered["index"]
    return game_id

@app.callback(
    Output("game-details-panel", "children"),
    Input("selected_game_id", "data"),
    prevent_initial_call=True
)
def update_game_details_panel(game_id):
    if not game_id:
        return html.Div("Select a game to view details.", style={"color": "#888", "padding": "1rem"})
    
    match = final_df[final_df["id"] == game_id]
    if match.empty:
        return html.Div("Game not found.", style={"color": "#888", "padding": "1rem"})
    
    row = match.iloc[0]
    return dbc.Card([
        dbc.CardHeader(
            html.Span(row.get("name", "Unnamed Game"), style={"fontWeight": "bold", "fontSize": "1.5rem"})
        ),
        html.Div([
            html.Img(
                src=row.get("background_image", ""),
                style={"width": "100%", "height": "200px", "objectFit": "cover", "borderRadius": "8px"}
            ) if row.get("background_image") else None,
            html.Hr(),
            html.P(f"Rating: {row.get('rating', 'N/A')}"),
            html.P(f"Metacritic: {row.get('metacritic', 'N/A')}"),
            html.P(f"Released: {row.get('released', 'Unknown')}"),
            html.P(f"Added by users: {row.get('added', 'N/A')}"),
            html.P(f"YouTube count: {row.get('youtube_count', 'N/A')}"),
            html.P(f"Twitch count: {row.get('twitch_count', 'N/A')}"),
            html.P(
                (row.get("description_raw") or "")[:300] + "..." if isinstance(row.get("description_raw"), str) else "No description available.",
                style={"fontSize": "0.9rem"}
            )
        ], style={"padding": "1rem"})
    ], style={"marginBottom": "1rem"})



# =========================
# Run server
# =========================
def get_free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

PORT = get_free_port()

def open_browser():
    webbrowser.open_new(f"http://127.0.0.1:{PORT}")

if __name__ == "__main__":
    threading.Timer(1, open_browser).start()
    app.run(debug=True, use_reloader=False, port=PORT)

